In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("test")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/24 14:08:29 WARN Utils: Your hostname, CrisBook.local, resolves to a loopback address: 127.0.0.1; using 10.190.175.101 instead (on interface en0)
26/04/24 14:08:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 14:08:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/24 14:08:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/24 14:08:30 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/24 14:08:30 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
nvd_df = spark.read.parquet("../data/dev/silver/nvd")
kev_df = spark.read.parquet("../data/dev/silver/kev")
epss_df = spark.read.parquet("../data/dev/silver/epss")

In [3]:
print("NVD rows:", nvd_df.count())
print("KEV rows:", kev_df.count())
print("EPSS rows:", epss_df.count())

NVD rows: 95979
KEV rows: 1579
EPSS rows: 328654


In [4]:
from pyspark.sql.functions import col, upper, trim

nvd_df = nvd_df.withColumn("cve_id", upper(trim(col("cve_id"))))
kev_df = kev_df.withColumn("cve_id", upper(trim(col("cve_id"))))
epss_df = epss_df.withColumn("cve_id", upper(trim(col("cve_id"))))

In [5]:
nvd_selected = nvd_df.select(
    "cve_id",
    "description",
    "cwe",
    "published",
    "lastModified",
    "cvss_score",
    "cvss_severity"
)

kev_selected = kev_df.select(
    "cve_id",
    "kev_date_added",
    "required_action",
    "known_ransomware_campaign_use"
)

epss_selected = epss_df.select(
    "cve_id",
    "epss_score",
    "epss_percentile"
)

In [6]:
master_df = (
    nvd_selected
    .join(kev_selected, on="cve_id", how="left")
    .join(epss_selected, on="cve_id", how="left")
)

In [8]:
from pyspark.sql.functions import when, lit, coalesce
from pyspark.sql.types import DoubleType

master_df = (
    master_df
    .withColumn(
        "is_kev",
        when(col("kev_date_added").isNotNull(), lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "epss_score",
        coalesce(col("epss_score").cast(DoubleType()), lit(0.0))
    )
    .withColumn(
        "epss_percentile",
        coalesce(col("epss_percentile").cast(DoubleType()), lit(0.0))
    )
    .withColumn(
        "cvss_score",
        col("cvss_score").cast(DoubleType())
    )
)

In [9]:
master_df.show(10, truncate=False)

+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+-----------------------+-----------------------+----------+-------------+--------------+---------------+-----------------------------+----------+---------------+------+
|cve_id        |description                                                                                                                                                                                                                                                                           

In [7]:
print("Master rows:", master_df.count())
print("Distinct CVEs:", master_df.select("cve_id").distinct().count())

Master rows: 95979
Distinct CVEs: 95979


In [10]:
master_df.groupBy("is_kev").count().show()

+------+-----+
|is_kev|count|
+------+-----+
|     1|  379|
|     0|95600|
+------+-----+



In [11]:
master_df.orderBy(col("epss_score").desc()).select(
    "cve_id",
    "cvss_score",
    "cvss_severity",
    "epss_score",
    "epss_percentile",
    "is_kev",
    "description"
).show(20, truncate=False)

+--------------+----------+-------------+----------+---------------+------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
master_df.write.mode("overwrite").parquet("../data/dev/gold/master_vulnerabilities")

In [ ]:
master_check = spark.read.parquet("../data/dev/gold/master_vulnerabilities")

master_check.show(5, truncate=False)

+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-----------------------+-----------------------+----------+-------------+--------------+---------------+-----------------------------+----------+---------------+------+
|cve_id        |description                                                                                                                                                                                                                                                                                                                                             |cwe   |published              |lastModified           |cvss_score|cvss_severity|kev_date_added|require

26/04/24 17:19:19 WARN TransportChannelHandler: Exception in connection from 10.190.175.101/10.190.175.101:63117
io.netty.channel.unix.Errors$NativeIoException: readAddress(..) failed with error(-60): Operation timed out
